In [1]:
import pandas as pd

In [26]:
excel_file_path = r"C:\Users\vamsh\Downloads\TA MV2\python_server\ML_Training_datasets\5S_400Window_100Step_100h_10th.xlsx"
df = pd.read_excel(excel_file_path)

In [43]:
df.columns

Index(['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
       'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
       'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
       'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_value',
       'supertrend_trend', 'cci', 'roc', 'momentum3', 'extreme_type',
       'time_to_extreme', 'change_percentage', 'volume_sum', 'volume_avg',
       'volume_rel', 'volume_max_rel', 'fixed_h_pct_change',
       'fixed_h_direction', 'rel_high', 'rel_low', 'range_pct', 'subblock_len',
       'open_rel', 'high_rel', 'low_rel', 'close_rel', 'pct_change_1',
       'pct_change_2', 'hour', 'minute', 'weekday', 'timestamp'],
      dtype='object')

In [44]:
df.shape

(78276, 50)

In [45]:
counts = df['extreme_type'].value_counts()
counts2 = df['fixed_h_direction'].value_counts()
print(counts)
print(counts2)

stable    33599
up        25218
down      19459
Name: extreme_type, dtype: int64
stable    51559
up        15172
down      11545
Name: fixed_h_direction, dtype: int64


In [46]:

def balance_memecoin_dataset(df, random_state=42):
    """
    Downsample the majority 'stable' class in both extreme_type and fixed_h_direction
    to create a more balanced dataset for ML.
    
    Parameters:
        df : pd.DataFrame
            Original dataset with 'extreme_type' and 'fixed_h_direction' columns
        random_state : int
            Seed for reproducibility
    
    Returns:
        df_balanced : pd.DataFrame
            Balanced dataset
    """

    # -----------------------
    # Step 1: Balance extreme_type
    # -----------------------
    up_down_rows = df[df['extreme_type'].isin(['up','down'])]
    stable_rows_extreme = df[df['extreme_type'] == 'stable']

    # Target stable count = average of up/down
    target_stable_count_extreme = int(len(up_down_rows) / 2)
    stable_sampled_extreme = stable_rows_extreme.sample(n=target_stable_count_extreme,
                                                        random_state=random_state)

    df_extreme_balanced = pd.concat([up_down_rows, stable_sampled_extreme])

    # -----------------------
    # Step 2: Balance fixed_h_direction
    # -----------------------
    up_down_rows_h = df_extreme_balanced[df_extreme_balanced['fixed_h_direction'].isin(['up','down'])]
    stable_rows_h = df_extreme_balanced[df_extreme_balanced['fixed_h_direction'] == 'stable']

    # Target stable count = average of up/down
    target_stable_count_h = int(len(up_down_rows_h) / 2)
    stable_sampled_h = stable_rows_h.sample(n=target_stable_count_h,
                                           random_state=random_state)

    df_balanced = pd.concat([up_down_rows_h, stable_sampled_h]).sample(frac=1,
                                                                       random_state=random_state).reset_index(drop=True)

    return df_balanced


In [47]:
df_new = balance_memecoin_dataset(df)

In [48]:
df_new.shape

(38017, 50)

In [49]:
counts = df_new['extreme_type'].value_counts()
counts2 = df_new['fixed_h_direction'].value_counts()
print(counts)
print(counts2)

up        17650
down      11705
stable     8662
Name: extreme_type, dtype: int64
up        14317
stable    12672
down      11028
Name: fixed_h_direction, dtype: int64


In [50]:
# df.drop()

In [51]:
# Shuffle dataset
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [52]:
counts = df_shuffled['extreme_type'].value_counts()
counts2 = df_shuffled['fixed_h_direction'].value_counts()
print(counts)
print(counts2)

stable    33599
up        25218
down      19459
Name: extreme_type, dtype: int64
stable    51559
up        15172
down      11545
Name: fixed_h_direction, dtype: int64


In [53]:
df_shuffled.columns

Index(['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
       'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
       'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
       'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_value',
       'supertrend_trend', 'cci', 'roc', 'momentum3', 'extreme_type',
       'time_to_extreme', 'change_percentage', 'volume_sum', 'volume_avg',
       'volume_rel', 'volume_max_rel', 'fixed_h_pct_change',
       'fixed_h_direction', 'rel_high', 'rel_low', 'range_pct', 'subblock_len',
       'open_rel', 'high_rel', 'low_rel', 'close_rel', 'pct_change_1',
       'pct_change_2', 'hour', 'minute', 'weekday', 'timestamp'],
      dtype='object')

In [40]:
df_shuffled['time_to_extreme'].describe()

count    38017.000000
mean      1959.084752
std        138.495894
min       1060.000000
25%       1995.000000
50%       1995.000000
75%       1995.000000
max       1995.000000
Name: time_to_extreme, dtype: float64

In [41]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import classification_report, mean_absolute_error
# from xgboost import XGBClassifier, XGBRegressor
# from sklearn.utils import shuffle

# # ------------------------------
# # 0. Shuffle dataset
# # ------------------------------
# # df_shuffled = df_new.sample(frac=1, random_state=42).reset_index(drop=True)

# # ------------------------------
# # 1. Feature Engineering: lagged / rolling features
# # ------------------------------
# df_shuffled['rsi_1'] = df_shuffled['rsi'].shift(1)
# df_shuffled['rsi_3'] = df_shuffled['rsi'].rolling(3).mean()
# df_shuffled['rsi_5'] = df_shuffled['rsi'].rolling(5).mean()

# df_shuffled['atr_3'] = df_shuffled['atr'].rolling(3).mean()
# df_shuffled['atr_5'] = df_shuffled['atr'].rolling(5).mean()

# df_shuffled['volume_avg_3'] = df_shuffled['volume_avg'].rolling(3).mean()
# df_shuffled['volume_avg_5'] = df_shuffled['volume_avg'].rolling(5).mean()
# df_shuffled['obv_3'] = df_shuffled['obv'].rolling(3).mean()

# df_shuffled.fillna(method='bfill', inplace=True)

# # Encode categorical feature once
# le_supertrend = LabelEncoder()
# df_shuffled['supertrend_trend_enc'] = le_supertrend.fit_transform(df_shuffled['supertrend_trend'])

# # Encode direction for hybrid use
# le_dir = LabelEncoder()
# df_shuffled['extreme_type_enc'] = le_dir.fit_transform(df_shuffled['extreme_type'])

# # ------------------------------
# # 2. Feature columns
# # ------------------------------
# feature_cols = ['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
#        'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
#        'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
#        'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_value',
#        'cci', 'roc', 'momentum3', 'volume_sum', 
#        'volume_avg', 'volume_rel', 'volume_max_rel',
#        'rel_high', 'rel_low', 'range_pct', 'subblock_len',
#        'open_rel', 'high_rel', 'low_rel', 'close_rel', 'pct_change_1',
#        'pct_change_2', 'hour', 'minute', 'weekday']



# X = df_shuffled[feature_cols]

# # ------------------------------
# # 3. Targets
# # ------------------------------
# y_dir = df_shuffled['extreme_type_enc']                 # classifier
# y_pct = df_shuffled['change_percentage']               # regression

# # Log-transform % change to reduce outliers
# y_pct_log = np.log1p(np.abs(y_pct)) * np.sign(y_pct)

# # ------------------------------
# # 4. Train-test split
# # ------------------------------
# X_train_dir, X_test_dir, y_train_dir, y_test_dir = train_test_split(
#     X, y_dir, test_size=0.2, random_state=42, stratify=y_dir
# )

# # ------------------------------
# # 5. Train Direction Classifier
# # ------------------------------
# clf_dir = XGBClassifier(
#     n_estimators=800,
#     learning_rate=0.05,
#     max_depth=10,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     eval_metric="mlogloss",
#     use_label_encoder=False,
#     random_state=42
# )
# clf_dir.fit(X_train_dir, y_train_dir)
# y_pred_dir = clf_dir.predict(X_test_dir)

# print("=== Direction Classification ===")
# print(classification_report(y_test_dir, y_pred_dir))

# # Using booster scores (gain)
# importance_dict = clf_dir.get_booster().get_score(importance_type='gain')

# # Sort by importance descending
# importance_sorted = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

# print("=== Feature Importances (Gain) ===")
# for feat, score in importance_sorted:
#     print(f"{feat:30s} : {score:.4f}")

# # Or using sklearn API attribute (normalized to 0-1)
# importances = clf_dir.feature_importances_
# feat_imp = list(zip(feature_cols, importances))
# feat_imp_sorted = sorted(feat_imp, key=lambda x: x[1], reverse=True)

# print("\n=== Feature Importances (sklearn API) ===")
# for feat, score in feat_imp_sorted:
#     print(f"{feat:30s} : {score:.4f}")


# # ------------------------------
# # 6. Train % Change Regressor (Hybrid)
# # ------------------------------
# # Only predict for up/down moves
# mask_up_down = df_shuffled['extreme_type'] != 'stable'
# X_up_down = X[mask_up_down].copy()
# y_up_down_pct_log = y_pct_log[mask_up_down]

# # Add classifier prediction as feature
# X_up_down['pred_dir_feature'] = le_dir.transform(df_shuffled['extreme_type'][mask_up_down])

# X_train_pct, X_test_pct, y_train_pct, y_test_pct = train_test_split(
#     X_up_down, y_up_down_pct_log, test_size=0.2, random_state=42
# )

# reg_pct = XGBRegressor(
#     n_estimators=800,
#     learning_rate=0.05,
#     max_depth=10,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42
# )
# reg_pct.fit(X_train_pct, y_train_pct)

# # Predict hybrid: use classifier output + regressor
# y_pred_pct_hybrid = np.zeros(len(X_test_dir))
# for idx, row in enumerate(X_test_dir.index):
#     direction = y_pred_dir[idx]
#     if direction != le_dir.transform(['stable'])[0]:
#         # For predicted up/down, use regressor prediction
#         X_row = X_test_dir.loc[[row]].copy()
#         X_row['pred_dir_feature'] = direction
#         y_pred_pct_hybrid[idx] = reg_pct.predict(X_row)[0]
#     else:
#         y_pred_pct_hybrid[idx] = 0.0  # stable

# # Reverse log-transform
# y_pred_pct_hybrid_inv = np.sign(y_pred_pct_hybrid) * (np.expm1(np.abs(y_pred_pct_hybrid)))
# y_test_pct_actual = y_pct.loc[X_test_dir.index]

# # Evaluate regression
# mae_pct_hybrid = mean_absolute_error(y_test_pct_actual, y_pred_pct_hybrid_inv)
# print("=== % Change Regression (Hybrid) ===")
# print("MAE % Change:", mae_pct_hybrid)


In [54]:
df_shuffled.columns

Index(['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
       'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
       'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
       'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_value',
       'supertrend_trend', 'cci', 'roc', 'momentum3', 'extreme_type',
       'time_to_extreme', 'change_percentage', 'volume_sum', 'volume_avg',
       'volume_rel', 'volume_max_rel', 'fixed_h_pct_change',
       'fixed_h_direction', 'rel_high', 'rel_low', 'range_pct', 'subblock_len',
       'open_rel', 'high_rel', 'low_rel', 'close_rel', 'pct_change_1',
       'pct_change_2', 'hour', 'minute', 'weekday', 'timestamp'],
      dtype='object')

In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor
from sklearn.utils import shuffle

# ------------------------------
# 0. Shuffle dataset
# ------------------------------
# df_shuffled = df_new.sample(frac=1, random_state=42).reset_index(drop=True)

# ------------------------------
# 1. Feature Engineering: lagged / rolling features
# ------------------------------
df_shuffled['rsi_1'] = df_shuffled['rsi'].shift(1)
df_shuffled['rsi_3'] = df_shuffled['rsi'].rolling(3).mean()
df_shuffled['rsi_5'] = df_shuffled['rsi'].rolling(5).mean()

df_shuffled['atr_3'] = df_shuffled['atr'].rolling(3).mean()
df_shuffled['atr_5'] = df_shuffled['atr'].rolling(5).mean()

df_shuffled['volume_avg_3'] = df_shuffled['volume_avg'].rolling(3).mean()
df_shuffled['volume_avg_5'] = df_shuffled['volume_avg'].rolling(5).mean()
df_shuffled['obv_3'] = df_shuffled['obv'].rolling(3).mean()

df_shuffled.fillna(method='bfill', inplace=True)

# Encode categorical feature once
le_supertrend = LabelEncoder()
df_shuffled['supertrend_trend_enc'] = le_supertrend.fit_transform(df_shuffled['supertrend_trend'])

# Encode direction for hybrid use
le_dir = LabelEncoder()
df_shuffled['extreme_type_enc'] = le_dir.fit_transform(df_shuffled['extreme_type'])

# ------------------------------
# 2. Feature columns
# ------------------------------
feature_cols = ['rsi', 'ema10', 'ema20', 'ema50', 'ema100', 'ema200', 'ema_cross',
       'macd_line', 'macd_signal', 'macd_hist', 'vwap', 'adx', 'plus_di',
       'minus_di', 'stoch_k', 'stoch_d', 'boll_upper', 'boll_middle',
       'boll_lower', 'boll_percent', 'atr', 'obv', 'supertrend_value',
       'cci', 'roc', 'momentum3', 'volume_sum', 
       'volume_avg', 'volume_rel', 'volume_max_rel',
       'rel_high', 'rel_low', 'range_pct', 'subblock_len',
       'open_rel', 'high_rel', 'low_rel', 'close_rel', 'pct_change_1',
       'pct_change_2', 'hour', 'minute', 'weekday']



X = df_shuffled[feature_cols]

# ------------------------------
# 3. Targets
# ------------------------------
y_dir = df_shuffled['extreme_type_enc']                 # classifier
y_pct = df_shuffled['change_percentage']               # regression

# Log-transform % change to reduce outliers
y_pct_log = np.log1p(np.abs(y_pct)) * np.sign(y_pct)

# ------------------------------
# 4. Train-test split
# ------------------------------
X_train_dir, X_test_dir, y_train_dir, y_test_dir = train_test_split(
    X, y_dir, test_size=0.2, random_state=42, stratify=y_dir
)

# ------------------------------
# 5. Train Direction Classifier
# ------------------------------
clf_dir = XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42
)
clf_dir.fit(X_train_dir, y_train_dir)
y_pred_dir = clf_dir.predict(X_test_dir)

print("=== Direction Classification ===")
print(classification_report(y_test_dir, y_pred_dir))

# Using booster scores (gain)
importance_dict = clf_dir.get_booster().get_score(importance_type='gain')

# Sort by importance descending
importance_sorted = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

print("=== Feature Importances (Gain) ===")
for feat, score in importance_sorted:
    print(f"{feat:30s} : {score:.4f}")

# Or using sklearn API attribute (normalized to 0-1)
importances = clf_dir.feature_importances_
feat_imp = list(zip(feature_cols, importances))
feat_imp_sorted = sorted(feat_imp, key=lambda x: x[1], reverse=True)

print("\n=== Feature Importances (sklearn API) ===")
for feat, score in feat_imp_sorted:
    print(f"{feat:30s} : {score:.4f}")


# ------------------------------
# 6. Train % Change Regressor (Hybrid)
# ------------------------------
# Only predict for up/down moves
mask_up_down = df_shuffled['extreme_type'] != 'stable'
X_up_down = X[mask_up_down].copy()
y_up_down_pct_log = y_pct_log[mask_up_down]

# Add classifier prediction as feature
X_up_down['pred_dir_feature'] = le_dir.transform(df_shuffled['extreme_type'][mask_up_down])

X_train_pct, X_test_pct, y_train_pct, y_test_pct = train_test_split(
    X_up_down, y_up_down_pct_log, test_size=0.2, random_state=42
)

reg_pct = XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
reg_pct.fit(X_train_pct, y_train_pct)

# Predict hybrid: use classifier output + regressor
y_pred_pct_hybrid = np.zeros(len(X_test_dir))
for idx, row in enumerate(X_test_dir.index):
    direction = y_pred_dir[idx]
    if direction != le_dir.transform(['stable'])[0]:
        # For predicted up/down, use regressor prediction
        X_row = X_test_dir.loc[[row]].copy()
        X_row['pred_dir_feature'] = direction
        y_pred_pct_hybrid[idx] = reg_pct.predict(X_row)[0]
    else:
        y_pred_pct_hybrid[idx] = 0.0  # stable

# Reverse log-transform
y_pred_pct_hybrid_inv = np.sign(y_pred_pct_hybrid) * (np.expm1(np.abs(y_pred_pct_hybrid)))
y_test_pct_actual = y_pct.loc[X_test_dir.index]

# Evaluate regression
mae_pct_hybrid = mean_absolute_error(y_test_pct_actual, y_pred_pct_hybrid_inv)
print("=== % Change Regression (Hybrid) ===")
print("MAE % Change:", mae_pct_hybrid)


c:\users\vamsh\appdata\local\programs\python\python39\lib\site-packages\xgboost\core.py:158: UserWarning: [22:44:17] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


=== Direction Classification ===
              precision    recall  f1-score   support

           0       0.87      0.87      0.87      3892
           1       0.93      0.92      0.92      6720
           2       0.91      0.92      0.92      5044

    accuracy                           0.91     15656
   macro avg       0.90      0.90      0.90     15656
weighted avg       0.91      0.91      0.91     15656

=== Feature Importances (Gain) ===
range_pct                      : 185.0400
rel_low                        : 133.7707
rel_high                       : 126.9153
minus_di                       : 39.3833
ema10                          : 24.2202
macd_hist                      : 19.0199
cci                            : 18.1153
ema100                         : 17.9851
boll_percent                   : 17.8225
volume_avg                     : 17.5738
rsi                            : 16.9584
volume_rel                     : 16.4564
subblock_len                   : 16.0878
vwap           